# Privacy Risk Assessment and Data Retention Automation Toolkit: live demo
**ICDFA Group 4 · Capstone project**

This notebook runs the whole toolkit from start to finish, one stage at a time. Every stage has the same three notes:

- **Why it matters:** the privacy problem this step solves.
- **What it does:** what the command or code does.
- **Expected result:** what you should see, so the presenter can check the output against the slides.

**Important:** all data is **synthetic (fictional)**, the retention periods are **TEST POLICIES** (not legal requirements), and the analyst and reviewer in this demo are **simulated**. Nothing here is connected to a production system.

**Run the cells in order, top to bottom.** The report needs the register step first, and the disposal steps must run in the order propose, approve, run, recover.

| Stage | What it shows |
|---|---|
| 0 | Set up a clean copy |
| 1 | Discover possible personal data |
| 2 | Review findings and measure accuracy |
| 3 | Processing register |
| 4 | Privacy risk assessment |
| 5 | Data inventory |
| 6 | Retention review |
| 7 | Controlled disposal simulation |
| 8 | Management report |
| 9 | Automated tests |

## Stage 0: Set up a clean copy
**Why it matters:** a demo is only convincing if every result is produced live. Starting from an empty `output/` folder proves the results are not left over from an earlier run.

**What it does:** unzips the toolkit, moves into its folder, deletes any old results and lists the folder.

**Expected result:** the folders `answer_sheet`, `config`, `data`, `docs`, `output`, `src`, `tests` and the files `run.sh`, `README.md` and `pyproject.toml`.

*Before running:* upload `privacy_toolkit.zip` with the Files panel. Group members can instead clone the Git repository (see the commented line).

In [1]:
# Unpack the toolkit. Upload privacy_toolkit.zip in the Files panel first.
# (Group members can instead clone the repository:
#  !git clone https://github.com/victordeman/privacy-risk-retention-toolkit.git
#  %cd privacy-risk-retention-toolkit )
!unzip -q -o privacy_toolkit.zip
# Move into the toolkit folder so run.sh and the relative paths work.
%cd privacy_toolkit
# Delete old results so every output in this demo is produced live.
!rm -rf output/*
# List the folder to confirm the toolkit is in place.
!ls

/content/privacy_toolkit
answer_sheet  data  output	    README.md  src
config	      docs  pyproject.toml  run.sh     tests


## Stage 1: Discover possible personal data
**Why it matters:** an organisation cannot protect, restrict or delete personal data it does not know it holds. Discovery builds the first picture of where personal data sits.

**What it does:** lists the six source files, then scans every column with the pattern rules in `config/discovery_rules.json` (email, phone, name, ID number, IP address). Source files are read-only and hash-checked, and results show masked values only.

**Expected result:** `Scanned 6 files, 30 findings (29 medium/high). Skipped: 0.`

**Key message:** a pattern match is only *possible* personal data. It is not a confirmed classification until an analyst reviews it.

In [2]:
# Show the six synthetic source files: employees, customers, suppliers, tickets, marketing, logs.
!ls data/source
# Run discovery: scan every column against the documented rules; write masked findings to output/.
!bash run.sh discover

access_logs.csv  employees.csv		 suppliers.csv
customers.csv	 marketing_contacts.csv  support_tickets.csv
Scanned 6 files, 30 findings (29 medium/high). Skipped: 0.
Pattern matches indicate POSSIBLE personal information. A finding is not a confirmed classification until an analyst has reviewed it.


### Stage 1b: Inspect the findings
**Why it matters:** transparency. Anyone can see exactly which column was flagged, by which rule, with what confidence.

**What it does:** loads `output/discovery_findings.csv` into a table. Each row is one flagged column with its category, the rule that matched, the match rate, a confidence level and masked examples.

**Expected result:** 30 rows. Point out the two false positives that look like personal data but are not: `suppliers.company_name` (a business name, automatically lowered to *low* confidence by a negative hint) and `suppliers.category` (values like "Professional Services" look like names, so it needs analyst rejection).

In [3]:
# Load the findings table so the group can inspect each flagged column.
# Masked examples show only the shape of the value, never the full personal data.
import pandas as pd
pd.read_csv("output/discovery_findings.csv").head(30)

,finding_id,file,column,category,rule_matched,matched_records,non_empty_records,total_records,match_rate,column_name_hint,negative_hint,confidence,masked_examples,review_status
0,F-001,access_logs.csv,ip_address,ip_address,ipv4,70,70,70,1.000,Y,N,high,41.*.*.* ; 46.*.*.*,pending_review
1,F-002,access_logs.csv,user_id,identification_number,employee_id,50,70,70,0.714,Y,N,high,EMP**** ; EMP****,pending_review
2,F-003,customers.csv,customer_id,identification_number,customer_id,90,90,90,1.000,Y,N,high,CUS***** ; CUS*****,pending_review
3,F-004,customers.csv,email,email,email_address,90,90,90,1.000,Y,N,high,g***@example.com ; r***@test.org,pending_review
4,F-005,customers.csv,first_name,name,single_capitalised_word_in_name_column,90,90,90,1.000,Y,N,high,J*** ; R***,pending_review
5,F-006,customers.csv,last_name,name,single_capitalised_word_in_name_column,90,90,90,1.000,Y,N,high,G*** ; B***,pending_review
6,F-007,customers.csv,phone,phone,digits_10_plus|intl_spaced|na_formatted,90,90,90,1.000,Y,N,high,*********37 ; ********75,pending_review
7,F-008,customers.csv,tax_id,identification_number,tax_id,45,45,90,1.000,Y,N,high,TXS******* ; TXK*******,pending_review
8,F-009,employees.csv,email,email,email_address,70,70,70,1.000,Y,N,high,l***@company.com ; r***@company.com,pending_review
9,F-010,employees.csv,employee_id,identification_number,employee_id,70,70,70,1.000,Y,N,high,EMP**** ; EMP****,pending_review


## Stage 2: Analyst review and accuracy
**Why it matters:** detecting is not confirming. A person must confirm or reject each finding, and the team must show how accurate the rules are.

**What it does:** the first command records analyst decisions (confirm or reject, with a log). In this demo the analyst is **simulated**, so use `--interactive` locally to type your own decisions. The second command compares the findings with the labelled answer sheet of 28 personal-data columns and 8 explicit non-matches, then calculates precision and recall.

**Expected result:** 2 false positives rejected, 0 missed. Precision rises from **0.933** (raw rules) to **1.0** (after review), recall stays at **1.0**.

In [5]:
# Record analyst decisions (confirm / reject). SIMULATED here; a human analyst uses --interactive in a terminal.
!bash run.sh review --simulate --analyst "Chioma"
# Measure accuracy at column level: correct, false positives, missed, precision and recall.
!bash run.sh evaluate

Simulated analyst reviewed 30 findings (DEMO ONLY - stands in for a human).
Missed detections: 0; rejected (false positives): 2
raw_rules                                  TP=28 FP=2 missed=0 P=0.933 R=1.0 F1=0.966
actionable_confidence_medium_or_high       TP=28 FP=1 missed=0 P=0.966 R=1.0 F1=0.982
after_analyst_review                       TP=28 FP=0 missed=0 P=1.0 R=1.0 F1=1.0


## Stage 3: Processing register
**Why it matters:** an organisation must be able to say why it holds data, who owns it and how that has changed. A register with history gives accountability.

**What it does:** records an owner review of activity PA-001 (HR administration), which raises its version to 2 and stores old value, new value, reviewer, date and reason in a history file. The second command prints a summary of the register.

**Expected result:** `PA-001 reviewed by Chioma; now version 2` and `12 activities, 11 unreviewed`. The unreviewed activities appear later in the management report.

*Run this before the report.* The report reads the register and fails without it.

In [6]:
# Record an owner review of PA-001; the change is versioned in processing_register_history.csv.
!bash run.sh register --activity PA-001 --reviewer "Chioma" --reason "Annual owner review"
# Print a register summary: how many activities exist and how many are still unreviewed.
!bash run.sh register

PA-001 reviewed by Chioma; now version 2
12 activities, 11 unreviewed


## Stage 4: Privacy risk assessment
**Why it matters:** privacy risk must be judged in a consistent, documented way, and linked to the data it concerns, so the organisation can show how a risk was assessed and treated.

**What it does:** creates assessment **PRA-007** for activity PA-003 (supplier contract management). Risk = likelihood x severity (each 1 to 5). It records the safeguards, the mitigation, the residual risk after controls, an owner and a deadline, then validates the scores, owner and deadline.

**Expected result:** `Recorded PRA-007: inherent 9 (Medium), residual 4 (Low)` and `Assessment validation: OK`. The bands are 1-4 Low, 5-9 Medium, 10-15 High, 16-25 Critical.

In [7]:
# Create assessment PRA-007 for PA-003 (supplier contract management).
# Inherent risk = 3 x 3 = 9 (Medium). After the mitigation, residual = 2 x 2 = 4 (Low). Deadline 2027-03-31.
# The tool then validates the scores, owner and deadline.
!bash run.sh assess --activity PA-003 --assessor "Chioma" --harm "Supplier contact details exposed" --likelihood 3 --severity 3 --safeguards "Access limited to Finance and Legal" --measures "Quarterly access review" --residual-likelihood 2 --residual-severity 2 --deadline 2027-03-31

Recorded PRA-007: inherent 9 (Medium), residual 4 (Low)
Assessment validation: OK


## Stage 5: Data inventory
**Why it matters:** the inventory is the link that connects everything. Without it, discovery, register, risk and retention are four separate lists.

**What it does:** builds `data_inventory.csv` with one row per dataset or system, linking each to its processing activity, owner, sensitivity, retention rule and risk assessments. The code then shows the `suppliers.csv` row.

**Expected result:** `Inventory: 18 entries (6 files, 12 systems)`. The `suppliers.csv` row (DS-005) links to PA-003, Procurement Lead, retention rule RR-003 and assessment PRA-007. Scroll the table to the right to see those columns.

In [8]:
# Build the inventory: 6 files + 12 systems, each linked to activity, owner, retention rule and assessment.
!bash run.sh inventory
# Load the inventory table.
inv = pd.read_csv("output/data_inventory.csv")
# Show the suppliers.csv entry to demonstrate the links (DS-005 -> PA-003 -> RR-003 -> PRA-007).
inv[inv["name"] == "suppliers.csv"]

Inventory: 18 entries (6 files, 12 systems)


,inventory_id,type,name,storage_location,record_count,activity_id,business_purpose,responsible_owner,documented_data_categories,discovered_categories,recipients,sensitivity,sensitivity_basis,retention_rule_id,retention_policy,assessment_ids,scan_status
4,DS-005,file,suppliers.csv,/content/privacy_toolkit/data/source,40.0,PA-003,Manage supplier relationships and contracts,Procurement Lead,"Company name, contact name/email/phone, contra...","email, identification_number, name, phone","Finance, legal",High,"discovered: High if identification numbers, el...",RR-003,7 years after contract end (TEST POLICY),PRA-007,scanned


## Stage 6: Retention review
**Why it matters:** keeping personal data longer than necessary is a privacy risk. The toolkit finds records past their retention period without ever putting held records or records with missing dates at risk.

**What it does:** applies the rules in `config/retention_schedule_config.json` to all 410 records using the as-of date 2026-11-14. Review due date = trigger date + retention years. Each record gets one outcome: eligible for review, on hold, insufficient information, active with no trigger, or not yet due. The result is compared with the corrected answer sheet. The second line lists the output columns.

**Expected result:** `Retention vs corrected answer sheet: ALL MATCH` (all 30 cells).

*Note for questions:* the supplied answer sheet did not add up to the record counts, so the team produced a corrected sheet and documented the change.

In [9]:
# Run the retention engine on all 410 records with the rules in the config file (no code change needed).
# It compares the result with the corrected answer sheet and prints ALL MATCH if all 30 cells agree.
!bash run.sh retention
# Load the record-level results.
r = pd.read_csv("output/retention_review.csv")
# List the columns the review records for every record.
r.columns.tolist()

Retention vs corrected answer sheet: ALL MATCH


['dataset',
 'record_id',
 'trigger_column',
 'trigger_value',
 'trigger_state',
 'retention_years',
 'review_due_date',
 'days_past_due',
 'on_hold',
 'hold_reason',
 'category',
 'reason',
 'data_quality_flag']

### Stage 6b: Summary by dataset
**Why it matters:** managers need the big picture, not 410 rows.

**What it does:** counts the outcomes for each dataset, with totals.

**Expected result:** 104 eligible for review, 12 on hold (expired), 10 insufficient information, 143 active with no trigger, 141 not yet due, **410** in total. This matches slide 9.

In [10]:
# Count outcomes per dataset; margins=True adds the totals row and column.
pd.crosstab(r["dataset"], r["category"], margins=True)

category,ACTIVE_NO_TRIGGER,ELIGIBLE_FOR_REVIEW,INSUFFICIENT_INFO,NOT_YET_DUE,ON_HOLD_EXPIRED,All
dataset,,,,,,
access_logs.csv,0,45,1,21,3,70
customers.csv,70,5,0,14,1,90
employees.csv,47,2,0,19,2,70
marketing_contacts.csv,0,29,9,18,4,60
suppliers.csv,3,1,0,36,0,40
support_tickets.csv,23,22,0,33,2,80
All,143,104,10,141,12,410


### Stage 6c: One eligible record, explained
**Why it matters:** shows the calculation is simple and checkable.

**What it does:** lists three eligible records with their trigger date, retention years, review due date, days past due and hold status.

**Expected result:** for example, an employee terminated on 2019-05-04 with a 7-year period has a due date of 2026-05-04, which is 194 days past due and not on hold. Trigger date + years = due date, and the due date is on or before 2026-11-14, so the record is eligible.

In [11]:
# Show three eligible records so the calculation (trigger date + years = due date) can be explained.
r[r["category"] == "ELIGIBLE_FOR_REVIEW"].head(3)[["dataset","trigger_column","trigger_value","retention_years","review_due_date","days_past_due","on_hold"]]

,dataset,trigger_column,trigger_value,retention_years,review_due_date,days_past_due,on_hold
7,employees.csv,termination_date,2019-05-04,7,2026-05-04,194.0,N
31,employees.csv,termination_date,2018-03-20,7,2025-03-20,604.0,N
74,customers.csv,account_closed_date,2020-03-06,5,2025-03-06,618.0,N


### Stage 6d: Missing or invalid dates are flagged, never deleted
**Why it matters:** a blank date is not permission to delete. Treating it that way could destroy records that should be kept.

**What it does:** lists the records whose trigger date is blank, invalid or a placeholder such as "unknown", with the reason.

**Expected result:** 10 records (9 marketing contacts and 1 access-log record). They are flagged for investigation and are excluded from disposal.

In [12]:
# Show records with a missing or invalid trigger date: they are flagged for investigation, never proposed for deletion.
r[r["category"] == "INSUFFICIENT_INFO"][["dataset","record_id","trigger_value","trigger_state","reason"]]

,dataset,record_id,trigger_value,trigger_state,reason
287,marketing_contacts.csv,MKT5007,(blank),missing,Trigger date missing or invalid on a record th...
293,marketing_contacts.csv,MKT5013,(blank),missing,Trigger date missing or invalid on a record th...
295,marketing_contacts.csv,MKT5015,invalid,invalid,Trigger date missing or invalid on a record th...
305,marketing_contacts.csv,MKT5025,(blank),missing,Trigger date missing or invalid on a record th...
310,marketing_contacts.csv,MKT5030,(blank),missing,Trigger date missing or invalid on a record th...
316,marketing_contacts.csv,MKT5036,unknown,invalid,Trigger date missing or invalid on a record th...
332,marketing_contacts.csv,MKT5052,(blank),missing,Trigger date missing or invalid on a record th...
337,marketing_contacts.csv,MKT5057,(blank),missing,Trigger date missing or invalid on a record th...
338,marketing_contacts.csv,MKT5058,(blank),missing,Trigger date missing or invalid on a record th...
399,access_logs.csv,LOG6059,(blank),missing,Trigger date missing or invalid on a record th...


### Stage 6e: Held records are protected
**Why it matters:** records under a legal hold or investigation must never be disposed of, even when they are past their retention period.

**What it does:** lists the records that have passed their review date but are on hold, with the hold reason.

**Expected result:** 12 records, with reasons such as Litigation hold, Legal hold, Legal request and Security investigation. Three have no reason recorded, which the report also points out. All 12 are excluded from every disposal proposal.

In [13]:
# Show records past retention but on hold: they must stay out of every disposal proposal.
r[r["category"] == "ON_HOLD_EXPIRED"][["dataset","record_id","review_due_date","on_hold","hold_reason"]]

,dataset,record_id,review_due_date,on_hold,hold_reason
4,employees.csv,EMP1004,2025-01-09,Y,NaN
62,employees.csv,EMP1062,2024-06-03,Y,Litigation hold
134,customers.csv,CUST2064,2026-02-04,Y,Legal hold
222,support_tickets.csv,TKT4022,2025-08-18,Y,NaN
267,support_tickets.csv,TKT4067,2026-06-29,Y,Escalated to legal
301,marketing_contacts.csv,MKT5021,2025-08-16,Y,NaN
304,marketing_contacts.csv,MKT5024,2025-07-04,Y,Consent dispute
321,marketing_contacts.csv,MKT5041,2024-12-01,Y,Legal request
325,marketing_contacts.csv,MKT5045,2022-12-08,Y,Legal request
343,access_logs.csv,LOG6003,2026-07-14,Y,Security investigation


## Stage 7: Controlled disposal simulation
Disposal is **simulated only**: it works on a copy and never touches the source files. Run the four steps in order.

### 7a. Propose
**Why it matters:** deletion decisions should start from a clear, reviewable list.

**What it does:** builds a disposal proposal from the records that are eligible and not on hold.

**Expected result:** `Proposal DP-2026-001: 104 records; holds excluded.`

In [14]:
# Build the disposal proposal: eligible records only, holds and unknown dates excluded.
!bash run.sh dispose propose

Proposal DP-2026-001: 104 records; holds excluded.


### 7b. Check that no held record is in the proposal
**Why it matters:** independent evidence that the hold rule works.

**What it does:** compares the proposal with the list of held records.

**Expected result:** `104 proposed; 0 held records in proposal`.

In [15]:
# Independent check: count how many held record IDs appear in the proposal (should be 0).
p = pd.read_csv("output/disposal_proposal.csv")
held = r[r["category"] == "ON_HOLD_EXPIRED"]["record_id"]
print(len(p), "proposed;", p["record_id"].isin(held).sum(), "held records in proposal")

104 proposed; 0 held records in proposal


### 7c. Approve
**Why it matters:** no disposal without a named person accountable for the decision.

**What it does:** records an approval by a named reviewer, the date and a checksum of the proposal. If the proposal changes after approval, the run is refused.

**Expected result:** `APPROVED by Chioma on 2026-11-14: 104 records` (replace the name with your own when presenting).

In [16]:
# Record approval by a named reviewer, with the date and a checksum so the proposal cannot change afterwards.
!bash run.sh dispose approve --reviewer "Chioma"

APPROVED by Chioma on 2026-11-14: 104 records


### 7d. Run on a copy
**Why it matters:** the team can prove what a disposal would do without risking real data.

**What it does:** deletes the approved records in a separate working copy, re-checks the holds at execution, takes a recovery copy first and verifies that the source files are unchanged.

**Expected result:** `DELETED_IN_COPY: 104`, `BLOCKED_ON_HOLD: 0`, `source_unchanged: True`.

In [17]:
# Execute the approved disposal on a copy only. Holds are re-checked here; the source hash is verified.
!bash run.sh dispose run

{'run_id': 'RUN-001', 'counts': {'DELETED_IN_COPY': 104, 'BLOCKED_ON_HOLD': 0, 'NOT_FOUND': 0}, 'source_unchanged': True, 'working_copy': '/content/privacy_toolkit/output/disposal_simulation/working_copy', 'recovery_copy': '/content/privacy_toolkit/output/disposal_simulation/recovery_copy'}


### 7e. Recover and read the log
**Why it matters:** disposal must be reversible in the test environment and fully logged, so the organisation can evidence what happened.

**What it does:** restores the recovery copy and proves it is identical to the source. The second line shows the start of the simulation log: decision, reviewer, date and result for each record.

**Expected result:** `identical_to_source: True` for all 6 files with the original row counts (70, 90, 70, 60, 40, 80), then a log showing `APPROVED`, the reviewer and `DELETED_IN_COPY`.

In [18]:
# Restore the recovery copy and prove it is identical to the source (6 files, original row counts).
!bash run.sh dispose recover
# Show the start of the simulation log: every record's decision, reviewer, date and result.
pd.read_csv("output/simulation_log.csv").head()

{'identical_to_source': True, 'files_checked': 6, 'restored_row_counts': {'access_logs.csv': 70, 'customers.csv': 90, 'employees.csv': 70, 'marketing_contacts.csv': 60, 'suppliers.csv': 40, 'support_tickets.csv': 80}, 'mismatched_files': []}


,timestamp,run_id,event,dataset,record_id,decision,reviewer,decision_date,execution_result
0,2026-11-15T15:32:55,RUN-001,RUN_START,NaN,NaN,APPROVED,Chioma,2026-11-14,NaN
1,2026-11-15T15:32:55,RUN-001,RECORD,access_logs.csv,LOG6000,APPROVED,Chioma,2026-11-14,DELETED_IN_COPY
2,2026-11-15T15:32:55,RUN-001,RECORD,access_logs.csv,LOG6001,APPROVED,Chioma,2026-11-14,DELETED_IN_COPY
3,2026-11-15T15:32:55,RUN-001,RECORD,access_logs.csv,LOG6002,APPROVED,Chioma,2026-11-14,DELETED_IN_COPY
4,2026-11-15T15:32:55,RUN-001,RECORD,access_logs.csv,LOG6005,APPROVED,Chioma,2026-11-14,DELETED_IN_COPY


## Stage 8: Management report
**Why it matters:** managers need a readable summary of the privacy position, and it must not expose personal data itself.

**What it does:** generates `management_report.html` with counts, accuracy, inventory, unreviewed activities, risks, the retention review with calculations, the disposal result and the limitations. It shows rule names, counts and partially masked samples only. The display cell puts the report on a white background so it is readable in Colab's dark theme.

**Expected result:** `Report written: ...management_report.html`, then the report appears below. Key figures: 6 datasets, 410 records, 28 confirmed personal-data fields, 104 records awaiting a disposal decision, 12 on hold, 10 needing date investigation.

*Tip:* for presenting, download the HTML file from the Files panel (`output/`) and open it in a browser.

In [19]:
# Generate the masked HTML management report (counts and masked samples only).
!bash run.sh report
from IPython.display import HTML, display
# Read the report file.
html = open("output/management_report.html").read()
# Display it on a white background so the dark text is readable in Colab's dark theme.
display(HTML('<div style="background:#ffffff;padding:24px;border-radius:8px">' + html + '</div>'))

Report written: /content/privacy_toolkit/output/management_report.html


Category,Fields,Files,Field-level record matches
email,5,"customers.csv, employees.csv, marketing_contacts.csv, suppliers.csv, support_tickets.csv",340
identification_number,7,"access_logs.csv, customers.csv, employees.csv, suppliers.csv",405
ip_address,1,access_logs.csv,70
name,10,"customers.csv, employees.csv, marketing_contacts.csv, suppliers.csv, support_tickets.csv",670
phone,5,"customers.csv, employees.csv, marketing_contacts.csv, suppliers.csv, support_tickets.csv",264
File,Records,Records with at least one match,Fields flagged
access_logs.csv,70,70,2
customers.csv,90,90,6
employees.csv,70,70,7
marketing_contacts.csv,60,60,4


## Stage 9: Automated tests
**Why it matters:** repeatable results need proof. The tests show the toolkit behaves correctly on boundary dates, leap days, invalid tokens, tampered proposals, held records and unreadable files.

**What it does:** runs the full test suite (88 tests).

**Expected result:** `88 passed`. Four deliberate bugs (holds treated as eligible, blank closed dates treated as eligible, a proposal ignoring holds, and an off-by-one on the due date) were each caught by at least one test.

In [20]:
# Run all 88 automated tests (dates, holds, invalid input, tampering, recovery).
!bash run.sh test

........................................................................ [ 81%]
................                                                         [100%]
88 passed in 1.65s


## Summary for the group: what to remember

- **Detecting is not confirming.** A pattern match is only possible personal data. An analyst confirms or rejects each finding.
- **How holds prevent disposal.** Held records are excluded from the proposal and blocked again at execution.
- **A missing date is never permission to delete.** Those records are flagged for investigation.
- **How accuracy was measured.** At column level against 28 labelled personal-data columns: correct, false positive, missed.
- **Why the answer sheet was corrected.** Its counts did not add up to the 410 records, so a corrected sheet was produced and documented.
- **Limits.** Patterns cannot confirm identity, free text is not inspected, retention periods are test policies, and real use needs legal review.